# OpenSource Clipping — Autopilot on Kaggle (2x T4)

Runs `discover_new_clips.py` (poll approved channels -> clip -> upload to YouTube) on a Kaggle GPU notebook.

## One-time setup (do this in the Kaggle UI before running)

1. **Notebook settings -> Accelerator -> GPU T4 x2.**
2. **Add-ons -> Secrets**, add these (names must match exactly):
   - `GOOGLE_API_KEY` (required — Gemini)
   - `YOUTUBE_DATA_API_KEY` (optional — CC re-verification)
   - `HF_TOKEN` (optional — split-screen/diarization)
   - `PEXELS_API_KEY` (optional — B-roll)
3. **Create a Kaggle Dataset named `clipping-code`** containing this repo (zip your project folder and upload it as a Dataset, or upload the folder directly — Kaggle preserves structure). Add it as a **Data source** to this notebook (Add Data -> Your Datasets -> clipping-code).
4. **Create a Kaggle Dataset named `clipping-state`** containing (from your local machine):
   - `data/` (channel_trust.db, dedup store — your 565 approved channels)
   - `outputs/upload_history.json` (daily-cap tracking)
   - `.credentials/client_secret.json` and `.credentials/youtube_token.json` (already generated locally — OAuth login can't happen interactively on Kaggle)
   - `upload_safety.json`
   Add it as a **Data source** too (Add Data -> Your Datasets -> clipping-state).
5. After each run, **Save Version** (commit) so `/kaggle/working` (the updated DB/history/token) becomes this notebook's own output. For the *next* run, add **this notebook's own previous output** as an extra data source (Add Data -> Your Work -> this notebook) so state carries forward automatically instead of you re-uploading `clipping-state` by hand every time.

Kaggle has no built-in cron — each run is a manual (or externally triggered) "Save & Run All".

## 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

## 2. Copy code + prior state into the writable working directory

In [ ]:
import os, shutil, glob

WORKDIR = "/kaggle/working/app"

# --- Code: find the clipping-code dataset under /kaggle/input at any depth, then confirm
# it's the actual repo root (not an extra wrapper folder from how the zip was made) by
# checking for a marker file like requirements.txt. ---
search_paths = [p for p in glob.glob("/kaggle/input/**", recursive=True) if os.path.isdir(p)]
code_candidates = sorted(
    (p for p in search_paths if "code" in os.path.basename(p).lower()),
    key=lambda p: p.count(os.sep),
)
assert code_candidates, f"No dataset with 'code' in its name found. Searched: {search_paths} — add the clipping-code Dataset as a data source first."

def is_repo_root(p):
    return os.path.exists(os.path.join(p, "requirements.txt")) and os.path.exists(os.path.join(p, "main.py"))

verified = [p for p in code_candidates if is_repo_root(p)]
if verified:
    CODE_SRC = verified[0]
else:
    # Marker files not found directly in any candidate — search one level deeper inside
    # each candidate for the real root (handles a zip that wrapped its own folder).
    nested = [os.path.join(p, d) for p in code_candidates for d in os.listdir(p) if os.path.isdir(os.path.join(p, d))]
    nested_verified = [p for p in nested if is_repo_root(p)]
    assert nested_verified, f"Found 'code' folders but none contain requirements.txt/main.py directly or one level deep: {code_candidates}"
    CODE_SRC = nested_verified[0]

print("Using code source:", CODE_SRC)

if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
shutil.copytree(CODE_SRC, WORKDIR)

os.chdir(WORKDIR)
print("cwd:", os.getcwd())
print(os.listdir(".")[:20])

In [ ]:
# --- Prior state: find known state files anywhere under /kaggle/input by filename
# (robust to however the clipping-state dataset was zipped/nested) and place them
# where the pipeline expects them relative to WORKDIR. ---
import os, shutil, glob

for d in ["data", "outputs", ".credentials"]:
    os.makedirs(os.path.join(WORKDIR, d), exist_ok=True)

FILE_DESTINATIONS = {
    "channel_trust.db": "data/channel_trust.db",
    "processed.db": "data/processed.db",
    "client_secret.json": ".credentials/client_secret.json",
    "youtube_token.json": ".credentials/youtube_token.json",
    "upload_safety.json": "upload_safety.json",
    "upload_history.json": "outputs/upload_history.json",
    "cookies.txt": "cookies.txt",
}

restored = []
for filename, rel_dest in FILE_DESTINATIONS.items():
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if not matches:
        continue
    src = matches[0]
    dst = os.path.join(WORKDIR, rel_dest)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy(src, dst)
    restored.append(rel_dest)

if restored:
    print("Restored:", restored)
else:
    print("WARNING: no prior state files found — starting with an empty trust DB / dedup store / upload history.")

missing = [f for f in FILE_DESTINATIONS if f not in [os.path.basename(r) for r in restored]]
if missing:
    print("Not found (will start fresh for these):", missing)

## 3. Secrets -> `.env`

In [ ]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

def get_secret(name):
    try:
        return user_secrets.get_secret(name)
    except Exception:
        return ""

env_vars = {
    "GOOGLE_API_KEY": get_secret("GOOGLE_API_KEY"),
    "YOUTUBE_DATA_API_KEY": get_secret("YOUTUBE_DATA_API_KEY"),
    "HF_TOKEN": get_secret("HF_TOKEN"),
    "PEXELS_API_KEY": get_secret("PEXELS_API_KEY"),
    "NVIDIA_API_KEY": get_secret("NVIDIA_API_KEY"),
    "GROQ_API_KEY": get_secret("GROQ_API_KEY"),
}

# GOOGLE_API_KEY is required by autopilot.py's entry check even when --ai-provider nvidia/groq is used.
assert env_vars["GOOGLE_API_KEY"], "GOOGLE_API_KEY secret is missing — add it under Add-ons > Secrets."

with open(".env", "w", encoding="utf-8") as f:
    for k, v in env_vars.items():
        f.write(f"{k}={v}\n")

print("Wrote .env with:", [k for k, v in env_vars.items() if v])

## 4. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 4a. Install Deno (required for yt-dlp to download from YouTube)

YouTube's "n" anti-bot challenge requires yt-dlp to run a small piece of JavaScript per video to get valid download URLs. Kaggle's base image has no JS runtime installed, so without this step every YouTube download fails with `n challenge solving failed` / `The page needs to be reloaded` (not a code bug -- an environment gap). yt-dlp looks for `deno` specifically by default.

In [ ]:
import subprocess, os

subprocess.run(
    "curl -fsSL https://deno.land/install.sh | sh",
    shell=True, check=True,
)

# The installer puts the binary under ~/.deno/bin -- add it to PATH for this
# kernel process so both `!python ...` shell-magic cells below and yt-dlp's
# own subprocess calls can find it.
deno_bin = os.path.expanduser("~/.deno/bin")
os.environ["PATH"] = deno_bin + os.pathsep + os.environ["PATH"]

result = subprocess.run(["deno", "--version"], capture_output=True, text=True)
print(result.stdout or result.stderr)


## 4b. (Optional) Start a local vLLM server for AI analysis instead of Gemini/Groq

Uses both T4s (tensor-parallel-size 2) to serve an open-weight model with an OpenAI-compatible API on `localhost:8000`, so `--ai-provider local` in the run cell below talks to it exactly like it talks to NVIDIA/Groq — no API key, no external rate limit, only bounded by your weekly Kaggle GPU-hour budget.

Run this section INSTEAD of the Groq/Gemini run cell in section 5 (pick one provider per run, don't run both — vLLM occupies both GPUs and will starve Whisper/face-tracking of VRAM if something else needs them at the same time). Skip this whole section if you just want Gemini/Groq.

In [ ]:
!pip install -q vllm

In [ ]:
import subprocess, sys, time, os

LOCAL_MODEL = "Qwen/Qwen2.5-32B-Instruct-AWQ"

# Launched via the modern `vllm serve` CLI (the OpenAI-compatible server's
# entrypoint moved here in recent vLLM releases -- the older
# `python -m vllm.entrypoints.openai.api_server` module path is deprecated/
# removed depending on version, and since `pip install vllm` above is
# unpinned, it's whatever the latest release ships). Launched as a
# background process so this cell returns immediately; the actual pipeline
# run (section 5b below) talks to it over HTTP once it reports healthy.
#
# start_new_session=True detaches this into its own process group (POSIX
# setsid). Without it, the server is a child of this Jupyter kernel process
# and inherits its process group -- interrupting the kernel (Kaggle's "Stop"
# button, or any Ctrl+C-equivalent) sends SIGINT to the whole group, killing
# vLLM even though nothing targeted it directly. This was confirmed on a real
# run: the server started, passed its own health check, then received
# SIGINT ~11 minutes later and shut down cleanly with no error of its own.
vllm_log_path = "/kaggle/working/vllm_server.log"
vllm_log = open(vllm_log_path, "w")
vllm_proc = subprocess.Popen(
    [
        "vllm", "serve", LOCAL_MODEL,
        "--tensor-parallel-size", "2",
        "--dtype", "float16",  # T4 (Turing, SM 7.5) has no native bf16 support
        "--max-model-len", "16384",
        "--gpu-memory-utilization", "0.90",
        "--port", "8000",
    ],
    stdout=vllm_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
print(f"vLLM server starting (pid={vllm_proc.pid}), logging to {vllm_log_path}")


In [ ]:
import time, urllib.request, urllib.error

# Loading + AWQ-dequantizing a 32B model across two T4s takes a few minutes --
# poll the server's own health endpoint rather than guessing a fixed sleep
# duration. Also checks the process is still alive on every tick, so a crash
# (bad CLI args, OOM, missing entrypoint, etc.) is caught within seconds
# instead of silently waiting out the full timeout.
HEALTH_URL = "http://localhost:8000/health"
MAX_WAIT_SECONDS = 600
POLL_INTERVAL_SECONDS = 10

def _dump_log_and_raise(reason):
    print(reason)
    print(f"--- last 4000 chars of {vllm_log_path} ---")
    try:
        with open(vllm_log_path) as f:
            tail = f.read()[-4000:]
        print(tail if tail.strip() else "(log file is empty)")
    except FileNotFoundError:
        print("(log file does not exist)")
    raise RuntimeError(reason)

start = time.time()
ready = False
while time.time() - start < MAX_WAIT_SECONDS:
    return_code = vllm_proc.poll()
    if return_code is not None:
        _dump_log_and_raise(f"vLLM server process exited early with code {return_code}.")

    try:
        with urllib.request.urlopen(HEALTH_URL, timeout=5) as resp:
            if resp.status == 200:
                ready = True
                break
    except (urllib.error.URLError, ConnectionError):
        pass
    elapsed = int(time.time() - start)
    print(f"  ...waiting for vLLM server ({elapsed}s elapsed)")
    time.sleep(POLL_INTERVAL_SECONDS)

if not ready:
    _dump_log_and_raise("vLLM server did not become healthy within the timeout.")

print("vLLM server is healthy and ready.")


## 5. Run the pipeline
Adjust `--max-new` to whatever fits your remaining weekly GPU-hour budget.

In [ ]:
!python discover_new_clips.py --max-new 5 --channel-poll-limit 50 --upload-youtube --whisper-device cuda --whisper-compute-type float16 --ai-provider groq --groq-model openai/gpt-oss-120b --cookies-file cookies.txt

## 5b. Run the pipeline using the local vLLM provider

Only run this if section 4b's vLLM server is up (section 5's Groq/Gemini cell is the alternative — use one or the other).

In [ ]:
!python discover_new_clips.py --max-new 5 --channel-poll-limit 50 --upload-youtube --whisper-device cuda --whisper-compute-type float16 --ai-provider local --local-model "Qwen/Qwen2.5-32B-Instruct-AWQ" --local-base-url "http://localhost:8000/v1" --cookies-file cookies.txt

## 6. Persist state back to `/kaggle/working` so it survives as this notebook's output

In [ ]:
import shutil, os

PERSIST_ROOT = "/kaggle/working"
for sub in ["data", "outputs", ".credentials"]:
    src = os.path.join(WORKDIR, sub)
    if os.path.exists(src):
        dst = os.path.join(PERSIST_ROOT, sub)
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)

safety_src = os.path.join(WORKDIR, "upload_safety.json")
if os.path.exists(safety_src):
    shutil.copy(safety_src, os.path.join(PERSIST_ROOT, "upload_safety.json"))

print("Persisted data/, outputs/, .credentials/, upload_safety.json to /kaggle/working")
print("Now click 'Save Version' (Save & Run All) so this becomes the notebook's committed output —")
print("next run, add this notebook's own output as a data source to carry state forward.")